# 03 — Linear Regression Baseline | IDX Exchange
**Week 4 deliverable**

**Goal:** Establish the simplest sensible benchmark for predicting `ClosePrice`. Features are property characteristics and location, not asking price or fields observed only during a sale. Evaluate with R² and error metrics on **validation** first. Once baseline decisions are frozen, the official **test** month can be evaluated once.

**Note:** This notebook needs the authorized CRMLSSold data; synthetic smoke-test results must never be presented as actual performance.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
RAW_DIR = ROOT / 'data' / 'raw'
OUTPUT_DIR = ROOT / 'outputs'
from src import avm
print('Project:', ROOT)
print('Input:', RAW_DIR)


### 1. Load real records and construct chronology

In [ ]:
raw, filenames = avm.load_raw(RAW_DIR)
clean, audit = avm.clean_properties(raw)
splits, split_info = avm.chronological_split(clean, as_of="2026-10-12")
bounded_splits, outlier_audit = avm.price_outlier_filter(splits)
print("Split:", split_info)
print("Outlier thresholds (from TRAIN ONLY):", outlier_audit)

### 2. Choose the input schema, excluding all target-leaking features

In [ ]:
numeric, categorical = avm.choose_features(bounded_splits["train"])
features = numeric + categorical
print("Numeric:",numeric)
print("Categorical:",categorical)
assert not set(features) & avm.FORBIDDEN_FEATURES

### 3. Fit a train-only, end-to-end scikit-learn Pipeline
`SimpleImputer`, `StandardScaler` and `OneHotEncoder` are inside `ColumnTransformer`. No preprocessing fits on validation or test.

In [ ]:
model = avm.build_pipeline(numeric, categorical)
X_train = bounded_splits["train"][features].copy()
import pandas as pd
X_train[numeric] = X_train[numeric].apply(pd.to_numeric, errors="coerce")
import numpy as np
for col in categorical:
    X_train[col] = X_train[col].astype(object).where(X_train[col].notna(), np.nan)
y_train = bounded_splits["train"]["ClosePrice"]
model.fit(X_train,y_train)
print("Fitted LinearRegression baseline")

### 4. Validate before touching the test data

In [ ]:
import pandas as pd
X_val = bounded_splits["validation"][features].copy()
X_val[numeric] = X_val[numeric].apply(pd.to_numeric,errors="coerce")
import numpy as np
for col in categorical:
    X_val[col] = X_val[col].astype(object).where(X_val[col].notna(), np.nan)
y_val = bounded_splits["validation"]["ClosePrice"]
val_predictions = model.predict(X_val)
val_metrics = avm.metrics(y_val, val_predictions)
display(pd.Series(val_metrics,name="validation"))

### 5. Save reusable pipeline and reports (optional full automation)

In [ ]:
report = avm.train_baseline(bounded_splits, OUTPUT_DIR, evaluate_test=False)
print("Validation metrics:",report["validation"])
print("Saved:", OUTPUT_DIR / "baseline_pipeline.joblib")

### Frozen baseline evaluation
Run locally with authorized data to reproduce evaluation reports. Saved results are excluded from this public copy.


In [ ]:
from src.verification import verify_frozen_baseline
summary, price_bands, sensitivity = verify_frozen_baseline(splits, bounded_splits, OUTPUT_DIR)
display(summary.round(4))
display(price_bands.round(2))
print("Unfiltered May sensitivity:", sensitivity)